# RQ7: What vehicle and market attributes best predict annual sales volume?

**Hypothesis:** Price, range, customer rating, and market segment are the strongest predictors of annual sales, with customer rating being the dominant factor.

**Methodology:**
1. Feature engineering + encoding of categorical variables
2. Pearson correlations with annual_sales_units
3. Ordinary Least Squares (OLS) multiple regression
4. Feature importance bar chart from standardized regression coefficients (PDF)
5. Correlation heatmap (PDF)
6. Regression results table (CSV)

In [ ]:
import pandas as pd, numpy as np, os, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.titlesize':12,'axes.labelsize':11,'figure.dpi':300,'axes.spines.top':False,'axes.spines.right':False})

for p in ['/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv','ev_market_2026.csv']:
    if os.path.exists(p): df = pd.read_csv(p); break

print('Shape:', df.shape)

In [ ]:
# Feature selection + encoding
features = ['price_usd','battery_capacity_kwh','range_miles','charging_speed_kw',
            'acceleration_0_60_mph','horsepower','safety_rating','autopilot_level',
            'seating_capacity','customer_rating','warranty_years']
target = 'annual_sales_units'

df_model = df[features + [target]].dropna()
print('Model dataset size:', len(df_model))

# Pearson correlations with target
corr_rows = []
for feat in features:
    r, p = pearsonr(df_model[feat], df_model[target])
    corr_rows.append({'Feature':feat, 'Pearson r':round(r,3), 'p-value':round(p,4)})
corr_df = pd.DataFrame(corr_rows).sort_values('Pearson r', key=abs, ascending=False)
print(corr_df.to_string(index=False))

In [ ]:
# Standardized OLS regression
X = df_model[features].values
y = df_model[target].values
scaler_X = StandardScaler(); scaler_y = StandardScaler()
X_std = scaler_X.fit_transform(X)
y_std = scaler_y.fit_transform(y.reshape(-1,1)).ravel()

reg = LinearRegression().fit(X_std, y_std)
coefs = pd.DataFrame({'Feature':features,'Beta':reg.coef_}).sort_values('Beta', key=abs, ascending=True)
r2 = reg.score(X_std, y_std)
print(f'OLS R² = {r2:.3f}')
print(coefs.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

# 1. Feature importance (horizontal bar)
colors = ['#c0392b' if b < 0 else '#2980b9' for b in coefs['Beta']]
ax1.barh(coefs['Feature'], coefs['Beta'], color=colors, height=0.6)
ax1.axvline(0, color='#333', linewidth=0.8)
ax1.set_xlabel('Standardized Regression Coefficient (β)')
ax1.set_title(f'Predictors of Annual EV Sales\n(OLS R²={r2:.3f})', fontsize=11)
from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color='#2980b9',label='Positive'), Patch(color='#c0392b',label='Negative')], fontsize=9, frameon=False)

# 2. Correlation heatmap (top 8 features by |r|)
top_feats = corr_df['Feature'].head(8).tolist()
corr_matrix = df_model[top_feats + [target]].corr()
mask = np.zeros_like(corr_matrix, dtype=bool)
np.fill_diagonal(mask, True)
sns.heatmap(corr_matrix, ax=ax2, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, annot_kws={'size':8}, vmin=-1, vmax=1)
ax2.set_title('Correlation Heatmap (Top Features)', fontsize=11)
ax2.tick_params(axis='x', rotation=40, labelsize=8.5)
ax2.tick_params(axis='y', labelsize=8.5)

plt.tight_layout()
fig.savefig('RQ7_Sales_Predictors.pdf', bbox_inches='tight', format='pdf')
plt.show(); print('Saved: RQ7_Sales_Predictors.pdf')

In [ ]:
# Save regression table
reg_tbl = pd.DataFrame({'Feature':features,'Std Beta':reg.coef_.round(4)}).sort_values('Std Beta', key=abs, ascending=False)
reg_tbl['Pearson r'] = [corr_df[corr_df['Feature']==f]['Pearson r'].values[0] for f in reg_tbl['Feature']]
reg_tbl['p-value (r)'] = [corr_df[corr_df['Feature']==f]['p-value'].values[0] for f in reg_tbl['Feature']]
reg_tbl.to_csv('RQ7_Regression_Table.csv', index=False)
print('Saved: RQ7_Regression_Table.csv'); reg_tbl